# Autoresearch: OZ Migration Component Detection

Visualization of experiment progress. Reads `results.tsv` and plots:
- Scatter plot of all experiments (kept vs discarded vs crashed)
- Running-maximum line showing best F1 over time

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [ ]:
results_path = Path("results.tsv")

if not results_path.exists() or results_path.stat().st_size == 0:
    print("No results.tsv found or file is empty. Run some experiments first.")
    df = pd.DataFrame(columns=["experiment", "status", "mean_f1", "description"])
else:
    df = pd.read_csv(
        results_path,
        sep="\t",
        header=None,
        names=["experiment", "status", "mean_f1", "description"],
    )

print(f"Total experiments: {len(df)}")
print(f"Kept: {(df['status'] == 'keep').sum()}")
print(f"Discarded: {(df['status'] == 'discard').sum()}")
print(f"Crashed: {(df['status'] == 'crash').sum()}")
if len(df) > 0:
    print(f"Best mean_f1: {df['mean_f1'].max():.6f}")
df.tail(10)

In [ ]:
if len(df) == 0:
    print("No data to plot.")
else:
    fig, ax = plt.subplots(figsize=(12, 6))

    colors = {"keep": "#22c55e", "discard": "#f97316", "crash": "#ef4444"}
    markers = {"keep": "o", "discard": "x", "crash": "^"}

    for status in ["crash", "discard", "keep"]:
        mask = df["status"] == status
        if mask.any():
            ax.scatter(
                df.loc[mask, "experiment"],
                df.loc[mask, "mean_f1"],
                c=colors.get(status, "#888"),
                marker=markers.get(status, "o"),
                label=status,
                s=60,
                alpha=0.8,
                zorder=3,
            )

    kept = df[df["status"] == "keep"].copy()
    if len(kept) > 0:
        kept = kept.sort_values("experiment")
        kept["running_max"] = kept["mean_f1"].cummax()
        ax.step(
            kept["experiment"],
            kept["running_max"],
            where="post",
            color="#16a34a",
            linewidth=2,
            label="running max (kept)",
            zorder=2,
        )

    ax.set_xlabel("Experiment #")
    ax.set_ylabel("Mean F1")
    ax.set_title("Autoresearch: Component Detection F1 Progress")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("progress.png", dpi=150)
    plt.show()
    print("Saved progress.png")

In [ ]:
if len(df) > 0:
    print("=== Kept experiments (improvements) ===")
    kept_df = df[df["status"] == "keep"].sort_values("experiment")
    for _, row in kept_df.iterrows():
        print(f"  #{int(row['experiment']):3d}  F1={row['mean_f1']:.6f}  {row['description']}")
    
    if len(kept_df) > 1:
        print(f"\nTotal improvement: {kept_df['mean_f1'].iloc[0]:.6f} -> {kept_df['mean_f1'].iloc[-1]:.6f}")